In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score

# 1. Load and clean the data
file_path = '../data/time_binned_dataset.csv'
print(f"Loading data: {file_path}")
df = pd.read_csv(file_path)

# Convert datetime and set as index
df['datetime'] = pd.to_datetime(df['datetime'])
df.set_index('datetime', inplace=True)

# Drop missing values
df = df.dropna()
print(f"After dropping missing values, the dataset has {len(df)} samples remaining.")

# 2. Separate features (X) and target variable (y)
target_col = 'ap_target'
X = df.drop(columns=[target_col])
y = df[target_col]

# [Optimization 2]: Apply log transformation to the target variable 
# (log1p handles 0 values safely)
y_log = np.log1p(y)

# 3. Split into train and test sets (maintain time order using shuffle=False)
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, shuffle=False)
# Keep the original y_test for final metric evaluation
_, _, _, y_test_original = train_test_split(X, y, test_size=0.2, shuffle=False)

# 4. [Optimization 1]: Use RobustScaler to handle extreme outliers in space weather data
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. [Optimization 3]: Tune SVR parameters
print("Training the optimized SVR model... (This may take several minutes for 40k+ samples, please wait)")
svm_model = SVR(kernel='rbf', C=10.0, gamma='scale', epsilon=0.1)
svm_model.fit(X_train_scaled, y_train_log)

# 6. Make predictions and revert the log transformation
y_pred_log = svm_model.predict(X_test_scaled)
# Convert the logarithmic predictions back to the original Ap index scale
y_pred = np.expm1(y_pred_log) 

# 7. Evaluate model performance
mse = mean_squared_error(y_test_original, y_pred)
mae = mean_absolute_error(y_test_original, y_pred)
r2 = r2_score(y_test_original, y_pred)

print("\n--- Optimized Model Evaluation Results ---")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

# --- Custom Accuracy Metrics ---

# Method 1: Tolerance-based Accuracy (Regression)
# Assuming a prediction is "correct" if it falls within ±5 of the actual Ap index
tolerance = 5.0  
accurate_predictions = np.abs(y_test_original - y_pred) <= tolerance
custom_accuracy = np.mean(accurate_predictions)
print(f"Custom Regression Accuracy (Tolerance ±{tolerance}): {custom_accuracy * 100:.2f}%")

# Method 2: Threshold-based Classification Accuracy
# Assuming Ap > 30 is considered a geomagnetic storm (1), otherwise normal (0)
threshold = 30.0  
y_test_binary = (y_test_original > threshold).astype(int)
y_pred_binary = (y_pred > threshold).astype(int)

storm_accuracy = accuracy_score(y_test_binary, y_pred_binary)
print(f"Geomagnetic Storm Classification Accuracy (Ap > {threshold}): {storm_accuracy * 100:.2f}%")

Loading data: ../data/time_binned_dataset.csv
After dropping missing values, the dataset has 43621 samples remaining.
Training the optimized SVR model... (This may take several minutes for 40k+ samples, please wait)

--- Optimized Model Evaluation Results ---
Mean Squared Error (MSE): 340.7480
Mean Absolute Error (MAE): 7.1599
R2 Score: -0.0676
Custom Regression Accuracy (Tolerance ±5.0): 67.82%
Geomagnetic Storm Classification Accuracy (Ap > 30.0): 94.18%


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score

def train_ap_forecast_model(file_path, horizon='3h', tolerance=5.0, threshold=30.0):
    """
    Trains an SVR model to forecast the Ap index for a specific time horizon.
    
    Parameters:
    - file_path: str, path to the dataset.
    - horizon: str, the time horizon to predict ('3h', '6h', '12h', or '24h').
    - tolerance: float, acceptable ± error for custom accuracy metric.
    - threshold: float, Ap index value considered to be a geomagnetic storm.
    
    Returns:
    - dict containing the trained model, scaler, and evaluation metrics.
    """
    print(f"\n{'='*50}")
    print(f"--- Building Model for {horizon} Horizon ---")
    print(f"{'='*50}")
    
    # 1. Load and clean the data
    df = pd.read_csv(file_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df.set_index('datetime', inplace=True)
    df = df.dropna()
    
    # 2. Prevent Data Leakage and Separate Features/Target
    target_col = f'ap_target_{horizon}'
    
    # List ALL target/future columns that must be dropped
    leakage_columns = [
        'ap_target_3h', 'storm_3h', 
        'ap_target_6h', 'storm_6h', 
        'ap_target_12h', 'storm_12h', 
        'ap_target_24h', 'storm_24h'
    ]
    
    if target_col not in leakage_columns:
        raise ValueError(f"Invalid horizon '{horizon}'. Choose from '3h', '6h', '12h', '24h'.")
        
    X = df.drop(columns=leakage_columns)
    y = df[target_col]
    
    # Apply log transformation to target
    y_log = np.log1p(y)
    
    # 3. Split into train and test sets (maintain time order)
    X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, shuffle=False)
    _, _, _, y_test_original = train_test_split(X, y, test_size=0.2, shuffle=False)
    
    # 4. Scale features using RobustScaler
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 5. Train SVR Model
    print(f"Training SVR model... (This may take a moment)")
    svm_model = SVR(kernel='rbf', C=10.0, gamma='scale', epsilon=0.1)
    svm_model.fit(X_train_scaled, y_train_log)
    
    # 6. Make predictions and revert the log transformation
    y_pred_log = svm_model.predict(X_test_scaled)
    y_pred = np.expm1(y_pred_log) 
    
    # 7. Evaluate model performance
    mse = mean_squared_error(y_test_original, y_pred)
    mae = mean_absolute_error(y_test_original, y_pred)
    r2 = r2_score(y_test_original, y_pred)
    
    # Custom Accuracies
    accurate_predictions = np.abs(y_test_original - y_pred) <= tolerance
    custom_accuracy = np.mean(accurate_predictions)
    
    y_test_binary = (y_test_original > threshold).astype(int)
    y_pred_binary = (y_pred > threshold).astype(int)
    storm_accuracy = accuracy_score(y_test_binary, y_pred_binary)
    
    # Print Output
    print(f"Metrics for {horizon} forecast:")
    print(f"  * MSE: {mse:.4f}")
    print(f"  * MAE: {mae:.4f}")
    print(f"  * R2 Score: {r2:.4f}")
    print(f"  * Custom Regression Accuracy (±{tolerance}): {custom_accuracy * 100:.2f}%")
    print(f"  * Storm Classification Accuracy (Ap > {threshold}): {storm_accuracy * 100:.2f}%")
    
    # Return useful artifacts so you can use them later
    return {
        'horizon': horizon,
        'model': svm_model,
        'scaler': scaler,
        'metrics': {
            'mse': mse,
            'mae': mae,
            'r2': r2,
            'regression_accuracy': custom_accuracy,
            'storm_accuracy': storm_accuracy
        }
    }

# ==========================================
# How to run the function for all horizons
# ==========================================

if __name__ == "__main__":
    file_path = '../data/time_binned_dataset.csv'
    
    # Define the horizons you want to forecast
    horizons_to_test = ['3h', '6h', '12h', '24h']
    
    # Dictionary to store all your trained models and their scalers
    trained_models = {}
    
    # Loop through and build a model for each one
    for h in horizons_to_test:
        # Run the function and save the dictionary output
        trained_models[h] = train_ap_forecast_model(file_path, horizon=h)
        
    print("\nAll models trained successfully!")
    
    # Example of how to access a specific model later:
    # my_12h_model = trained_models['12h']['model']
    # my_12h_scaler = trained_models['12h']['scaler']


--- Building Model for 3h Horizon ---
Training SVR model... (This may take a moment)
Metrics for 3h forecast:
  * MSE: 69197.3646
  * MAE: 12.6666
  * R2 Score: -153.3070
  * Custom Regression Accuracy (±5.0): 77.39%
  * Storm Classification Accuracy (Ap > 30.0): 95.77%

--- Building Model for 6h Horizon ---
Training SVR model... (This may take a moment)
Metrics for 6h forecast:
  * MSE: 1473.4550
  * MAE: 7.4873
  * R2 Score: -2.2855
  * Custom Regression Accuracy (±5.0): 73.86%
  * Storm Classification Accuracy (Ap > 30.0): 94.66%

--- Building Model for 12h Horizon ---
Training SVR model... (This may take a moment)
Metrics for 12h forecast:
  * MSE: 483.0097
  * MAE: 7.1581
  * R2 Score: -0.0771
  * Custom Regression Accuracy (±5.0): 71.20%
  * Storm Classification Accuracy (Ap > 30.0): 93.96%

--- Building Model for 24h Horizon ---
Training SVR model... (This may take a moment)
Metrics for 24h forecast:
  * MSE: 457.2069
  * MAE: 7.4106
  * R2 Score: -0.0197
  * Custom Regression 